1. 전략 아이디어 (Anomaly & Logic)

활용 Anomaly: 거시 경제 위기 국면에서의 '안전마진(Value)' 확보 및 2026년 상법 개정에 따른 '주주환원 프리미엄' 선점.

논리: ·고유가로 인한 시장 변동성 확대 시기에는 저PBR(가치) 종목의 하방 경직성이 강해짐.

단순 가치주가 아닌 ROE(수익성) 필터를 통해 '가치 함정'을 탈출한 우량주 선정.

자사주 소각 의무화 등 제도적 변화의 수혜를 직접적으로 입증할 수 있는 '총 주주환원율' 높은 기업에 집중하여 리레이팅 수익 추구.

2. 사용 데이터

Universe: 상장폐지 종목을 포함한 KOSPI/KOSDAQ 전 종목 

사용 데이터:  수정주가, 시가총액, 거래대금, PBR, ROE, 주주환원률

데이터 기간: 2014년 12월31일 ~ 2026년 2월28일

3. 전략 구성 방식 (Backtest Rules)

Step 1 (Liquidity): 최근 1개월 평균 거래대금 하위 20% 종목 제외

Step 2 (Quality Filter): ROE 5% 이상 또는 업종 내 ROE 상위 80% 종목 선별 

Step 3 (Value Ranking): PBR 하위 20% 종목 추출.

Step 4 (Shareholder Yield): 주주환원률 합계가 높은 상위 20종목 최종 선정.

Step 5 (Time-Lag Control): 모든 재무 데이터에 shift(3) (3개월 지연) 적용하여 공시 시차에 따른 선견 편향 원천 차단.

Step 6 (Execution): 월말 리밸런싱, 종목당 동일 가중 투자.



In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# 모든 데이터는 'Date'를 인덱스로, datetime 형식으로 로드
ac_df = pd.read_csv("수정주가.csv", index_col="Date", parse_dates=True)
mkt_cap_df = pd.read_csv("시가총액.csv", index_col='Date', parse_dates=True)
pbr_df = pd.read_csv("PBR.csv", index_col="Date", parse_dates=True)
roe_df = pd.read_csv("ROE.csv", index_col="Date", parse_dates=True)  # 가치함정 방지
yield_df = pd.read_csv("주주환원율.csv", index_col="Date", parse_dates=True) # 배당+소각
vol_df = pd.read_csv("거래대금.csv", index_col="Date", parse_dates=True) # 유동성 필터

In [ ]:
monthly_returns = ac_df.resample('ME').last().pct_change(fill_method=None)
monthly_mkt_cap = mkt_cap_df.resample('ME').last()
monthly_pbr = pbr_df.resample('ME').last()

#재무 데이터는 3개월 shift하여 공시 시차 반영
monthly_roe = roe_df.resample('ME').last().shift(3)
monthly_yield = yield_df.resample('ME').last().shift(3)
monthly_vol = vol_df.resample('ME').mean() # 한 달 평균 거래대금

monthly_pbr[monthly_pbr <= 0.2] = np.nan

In [ ]:
month_ends = ac_df.resample('ME').last().loc['2014-12-31':'2026-02-28'].index
portfolio_ret= pd.Series(dtype=float)
portfolio_ret[month_ends[0]] = 0.0

In [ ]:
for i in range(0, len(month_ends) - 1):
    start_date = month_ends[i]      # 리밸런싱 기준일 (t)
    end_date   = month_ends[i + 1]  # 다음 리밸런싱일 (t+1 수익률 확인)

    # Step.1 유동성 필터링 (거래대금 하위 20% 제외)
    valid_vol = monthly_vol.loc[start_date].dropna()
    vol_cutoff = valid_vol.quantile(0.2)
    liquid_univ = valid_vol[valid_vol > vol_cutoff].index

    # Step.2 가치 함정 방지 (ROE 5% 이상 우량주만)
    roe_filtered = monthly_roe.loc[start_date, liquid_univ].dropna()
    quality_univ = roe_filtered[roe_filtered >= 5.0].index

    # Step.3 저PBR 전략 (PBR 하위 20% 선택)
    pbr_filtered = monthly_pbr.loc[start_date, quality_univ].dropna()
    pbr_cutoff = pbr_filtered.quantile(0.2)
    value_univ = pbr_filtered[pbr_filtered <= pbr_cutoff].index

    # Step.4 주주환원 강화 종목 선택 (배당+소각 상위 20종목)
    # 2026년 상법개정 수혜주 타겟팅
    final_basket = monthly_yield.loc[start_date, value_univ].dropna().nlargest(20).index

    # Step.5 수익률 계산 (동일가중)
    returns = monthly_returns.loc[end_date, final_basket].mean()
    portfolio_ret.loc[end_date] = returns

In [ ]:
portfolio_nav = (1 + portfolio_ret).cumprod()
portfolio_nav

In [ ]:
portfolio_nav.plot()

In [ ]:
ks200_df = pd.read_csv('kospi200.csv', index_col='Date', parse_dates=True)

In [ ]:
ks200_montly = ks200_df.resample('ME').last()
ks200_montly = ks200_montly.loc[portfolio_nav.index]
ks200_montly = (ks200_montly / ks200_montly.iloc[0]) * portfolio_nav.iloc[0]
ks200_montly['Portfolio'] = portfolio_nav
ks200_montly.plot()


In [ ]:
portfolio_ret = portfolio_ret[1:]

years = (portfolio_nav.index[-1] - portfolio_nav.index[0]).days / 365.25
cagr  = (portfolio_nav.iloc[-1] / portfolio_nav.iloc[0]) ** (1 / years) - 1
print(f"CAGR: {cagr:.2%}")


volatility = portfolio_ret.std() * np.sqrt(12) 
print(f"Annualized Volatility: {volatility:.2%}")


risk_free_rate = (1 + 0.02) ** (1/12) - 1  
excess_returns = portfolio_ret - risk_free_rate
sharpe_ratio   = excess_returns.mean() / excess_returns.std() * np.sqrt(12) 
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")


rolling_max = portfolio_nav.cummax()
drawdowns = (portfolio_nav - rolling_max) / rolling_max
mdd = drawdowns.min()

print(f"MDD: {mdd:.2%}")